In [ ]:
!pip uninstall -y bitsandbytes transformers accelerate
!pip install -U bitsandbytes==0.46.1
!pip install -U git+https://github.com/huggingface/transformers.git
!pip install -U accelerate sentencepiece

Found existing installation: bitsandbytes 0.46.1
Uninstalling bitsandbytes-0.46.1:
  Successfully uninstalled bitsandbytes-0.46.1
Found existing installation: transformers 5.8.1
Uninstalling transformers-5.8.1:
  Successfully uninstalled transformers-5.8.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.46.1-py3-none-manylinux_2_24_x86_64.whl (72.9 MB)


  Cloning https://github.com/huggingface/transformers.git to /tmp/pip-req-build-bsjtqk8z
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers.git /tmp/pip-req-build-bsjtqk8z
ERROR: Operation cancelled by user
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
^C


In [2]:
!pip install -U transformers

  Using cached transformers-5.8.1-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.8.1-py3-none-any.whl (10.6 MB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
peft 0.19.1 requires accelerate>=0.21.0, which is not installed.


In [5]:
!pip install -U accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 8.1 MB/s eta 0:00:00


In [6]:
import bitsandbytes
import transformers
import torch
import accelerate

print(accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

1.13.0
bitsandbytes: 0.46.1
transformers: 5.8.1
CUDA available: True


In [3]:
from google.colab import userdata
from huggingface_hub import login

token = userdata.get('HF_TOKEN')

login(token)

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "google/gemma-4-E4B-it"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto"
)

print("Model loaded successfully!")

Loading tokenizer...


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

Loading model...


model.safetensors:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
def ask_gemma(prompt, max_tokens=256):

    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True
    ).to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_tokens,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
        repetition_penalty=1.1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id
    )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return response.strip()

In [ ]:
print(ask_gemma("Write Python code for a calculator"))

```python
def add(x, y):
  """Adds two numbers together."""
  return x + y

def subtract(x, y):
  """Subtracts one number from another."""
  return x - y

def multiply(x, y):
  """Multiplies two numbers."""
  return x * y

def divide(x, y):
  """Divides one number by another."""
  if y == 0:
    return "Division by zero error!"
  else:
    return x / y

while True:
  print("Select operation:")
  print("1. Add")
  print("2. Subtract")
  print("3. Multiply")
  print("4. Divide")
  print("5. Exit")

  choice = input("Enter choice(1/2/3/4/5): ")

  if choice in ('1', '2', '3', '4'):
    try:
      num1 = float(input("Enter first number: "))
      num2 = float(input("Enter second number: "))
    except ValueError:
      print("Invalid input. Please enter numbers only.")
      continue


In [11]:
import gradio as gr
import torch

model.eval()

chat_history = []

def vidyabot(message):

    global chat_history

    try:

        chat_history.append({
            "role": "user",
            "content": message
        })

        # smaller memory window
        chat_history = chat_history[-4:]

        inputs = tokenizer.apply_chat_template(
            chat_history,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True
        ).to(model.device)

        with torch.no_grad():

            outputs = model.generate(
                **inputs,
                max_new_tokens=40,      # IMPORTANT
                temperature=0.5,
                do_sample=True,
                top_p=0.9,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id
            )

        generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

        response = tokenizer.decode(
            generated_tokens,
            skip_special_tokens=True
        )

        chat_history.append({
            "role": "assistant",
            "content": response
        })

        del inputs
        del outputs
        torch.cuda.empty_cache()

        return response

    except Exception as e:

        torch.cuda.empty_cache()
        return f"Error: {str(e)}"


demo = gr.Interface(
    fn=vidyabot,
    inputs=gr.Textbox(
        lines=2,
        placeholder="Ask VidyaBot..."
    ),
    outputs="text",
    title="VidyaBot",
    description="Offline AI Tutor powered by Gemma 4 E4B"
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6ef3c5f68015a04df2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
